# Lección 4.7 — Líneas base: ¿le gano a "adivinar el promedio"?

> **Módulo 4 · Evaluación: ¿mi modelo es bueno?** · Nivel 2 — Aprendiz
> ⏱️ Tiempo estimado: 80–100 min (lección con ejercicios intercalados ≈ 55 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 4.1 (MAE, RMSE, R²), Lección 4.2 (exactitud, precisión, *recall*), Lección 4.6 (validación cruzada)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar qué es una **línea base** (*baseline*) y por qué **ningún número de evaluación significa nada sin ella**.
- Construir líneas base "tontas" para regresión y clasificación con `DummyRegressor` y `DummyClassifier`.
- Construir líneas base para **series de tiempo** (persistencia y estacional ingenua) y una **regla de experto**.
- Medir la **mejora relativa** (*skill score*) de un modelo respecto a su línea base.
- Detectar comparaciones injustas: líneas base demasiado débiles o medidas en condiciones distintas.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes, make_classification
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, recall_score

plt.rcParams["figure.figsize"] = (9, 4)
print("✅ Todo listo")

---
## 1. 📖 La intuición: el meteorólogo perezoso

Un meteorólogo presume: *"¡Mis pronósticos aciertan el 80 % de los días!"*. ¿Es bueno?

Depende. Imagina a su primo, que no sabe nada de meteorología y siempre dice **"mañana va a hacer lo mismo que hoy"**. En una ciudad donde el clima cambia poco, ese primo perezoso acierta… **el 85 % de los días**. 😬

De repente el 80 % ya no impresiona: el experto, con sus satélites y supercomputadoras, **lo hace peor que alguien que no hace nada**.

Eso es una **línea base** (*baseline*): la predicción más simple y razonable que podrías hacer **sin modelo**. Es la vara contra la que se mide todo lo demás.

> 🧠 **Regla de oro:** un número de evaluación solo significa algo **comparado con otro número**. "MAE = 12" no dice nada; "MAE = 12 cuando adivinar el promedio da 30" dice mucho.

### ✋ Ejercicio 1 — ¿Impresiona o no?
Para cada afirmación, piensa cuál sería la línea base "perezosa" y si el resultado impresiona:

1. *"Mi modelo predice si un correo es spam con 92 % de exactitud."* (En tu bandeja, el 95 % de los correos **no** es spam.)
2. *"Mi modelo predice si mañana subirá la bolsa con 52 % de exactitud."* (Sube aproximadamente la mitad de los días.)
3. *"Mi modelo predice la temperatura de mañana con un error promedio de 1.5 °C."* (Decir "igual que hoy" tiene un error promedio de 2.5 °C.)

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **No impresiona: es peor que no hacer nada.** Decir "nunca es spam" acierta el 95 %. El modelo, con 92 %, pierde contra la línea base.
2. **Depende de cuánto dinero ganes con ese 2 %.** La línea base (moneda al aire o "siempre sube") ronda el 50 %. Ganarle por 2 puntos en bolsa podría ser valioso… o ruido. Habría que verificar que la mejora es estable (Lección 4.6).
3. **Sí impresiona**: reduce el error de 2.5 °C a 1.5 °C, un 40 % menos que la línea base "igual que hoy".

La pregunta clave siempre es: **¿comparado con qué?**
</details>

---
## 2. 📖 Líneas base para regresión: adivinar el promedio

En regresión (predecir un número), la línea base más clásica es **predecir siempre el promedio** del entrenamiento, sin mirar ninguna característica.

scikit-learn la trae lista: `DummyRegressor` ("regresor tonto"). Su perilla `strategy` decide qué número constante predice:

| `strategy` | Predice siempre… |
|---|---|
| `"mean"` (por defecto) | el promedio del entrenamiento |
| `"median"` | la mediana del entrenamiento |
| `"quantile"` | un percentil que tú eliges (`quantile=0.9`) |
| `"constant"` | un número que tú eliges (`constant=...`) |

Vamos a probarla con datos médicos reales: 442 pacientes con diabetes; hay que predecir **qué tanto avanzará la enfermedad en un año** (un número entre 25 y 346) a partir de edad, índice de masa corporal, presión arterial y análisis de sangre.

In [ ]:
X, y = load_diabetes(return_X_y=True, as_frame=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=0)

base_media = DummyRegressor(strategy="mean").fit(X_tr, y_tr)
print("La línea base predice para TODOS los pacientes:", base_media.predict(X_te[:3]).round(1))
print(f"Promedio del entrenamiento: {y_tr.mean():.1f}")

### 🔮 Ejercicio 2 — El R² del perezoso
Recuerda la Lección 4.1: el **R²** compara tu modelo con… predecir el promedio. Antes de ejecutar:
1. ¿Qué R² tendrá la línea base `mean` en el **entrenamiento**?
2. ¿Y en **prueba**: exactamente ese valor, o algo muy parecido?

✍️ *Mis predicciones:*

In [ ]:
def evaluar(nombre, modelo):
    pred = modelo.predict(X_te)
    print(f"{nombre:<22} MAE prueba = {mean_absolute_error(y_te, pred):5.1f}   "
          f"R² entrenamiento = {r2_score(y_tr, modelo.predict(X_tr)):6.3f}   R² prueba = {r2_score(y_te, pred):6.3f}")

evaluar("Línea base (media)", base_media)
evaluar("Regresión lineal", LinearRegression().fit(X_tr, y_tr))

<details><summary><b>👀 Ver respuesta</b></summary>

1. **R² = 0.000 en entrenamiento**, exacto. El R² *se define* comparando con el promedio del conjunto que evalúas, así que predecir ese promedio da cero por construcción.
2. En prueba sale **≈ 0 (−0.000)**, no exactamente cero: el promedio del entrenamiento no es idéntico al promedio de prueba. Con datos nuevos, una línea base de promedio puede incluso dar R² ligeramente **negativo**.

Lección importante: **el R² ya es una comparación contra una línea base** (la del promedio). Por eso un R² de 0.36 significa "explico el 36 % de lo que el promedio no explicaba".

La regresión lineal baja el MAE de **58.3 a 45.1**. Mejora, pero predecir enfermedades es difícil: aun así se equivoca en ~45 puntos de progresión.
</details>

### ✏️ Ejercicio 3 — ¿Media o mediana?
Cambia la estrategia de la línea base a `"median"`. Antes de ejecutar, apuesta: ¿mejora el **MAE**? ¿Y el **R²**?

✍️ *Mi apuesta:*

In [ ]:
ESTRATEGIA = "median"   # ✏️ prueba "mean", "median" o "quantile" (agrega quantile=0.9)
evaluar(f"Línea base ({ESTRATEGIA})", DummyRegressor(strategy=ESTRATEGIA).fit(X_tr, y_tr))

<details><summary><b>👀 Ver respuesta</b></summary>

Con `"median"`: el **MAE mejora un poquito (58.3 → 58.3, de 58.31 a 58.25)** y el **R² empeora (≈0.000 → −0.038)**.

No es casualidad: la **mediana** es el número constante que minimiza el **error absoluto** (MAE), y la **media** es el que minimiza el **error cuadrático** (en el que se basan RMSE y R²). Si vas a medir con MAE, compara contra la mediana; si mides con RMSE o R², contra la media. **La línea base justa depende de la métrica.**
</details>

---
## 3. 📖 Líneas base para clasificación: la exactitud engañosa

En clasificación, la línea base más común es **predecir siempre la clase más frecuente**. Con datos desbalanceados (Lección 2.7) esto da exactitudes altísimas sin hacer nada.

`DummyClassifier` trae varias estrategias:

| `strategy` | Qué hace | Para qué sirve |
|---|---|---|
| `"most_frequent"` | Siempre dice la clase mayoritaria | La vara mínima de **exactitud** |
| `"prior"` | Igual, pero sus probabilidades son las proporciones de cada clase | Línea base para métricas de **probabilidad** (Lección 4.3) |
| `"stratified"` | Adivina al azar respetando las proporciones | Simula "adivinar con información de frecuencias" |
| `"uniform"` | Adivina al azar con todas las clases igual de probables | Una moneda al aire |

Caso: un equipo de soporte recibe 5,000 tickets y quiere detectar los **urgentes** para atenderlos primero. Solo una pequeña parte lo es.

In [ ]:
X_t, y_t = make_classification(n_samples=5000, n_features=8, n_informative=4,
                               weights=[0.96], flip_y=0.01, random_state=3)
X_t_tr, X_t_te, y_t_tr, y_t_te = train_test_split(X_t, y_t, stratify=y_t, random_state=0)
print(f"Tickets urgentes: {y_t.mean():.1%}")

### 🔮 Ejercicio 4 — El torneo de los tontos
Vamos a medir **exactitud** y **recall** de los urgentes (Lección 4.2) para las cuatro estrategias y para una regresión logística de verdad. Antes de ejecutar:
1. ¿Qué exactitud tendrá `most_frequent`?
2. ¿Qué recall tendrá `most_frequent`?
3. ¿Cuál de las estrategias "tontas" tendrá **mayor** recall?

✍️ *Mis predicciones:*

In [ ]:
modelos = {f"Dummy {s}": DummyClassifier(strategy=s, random_state=0)
           for s in ["most_frequent", "prior", "stratified", "uniform"]}
modelos["Regresión logística"] = LogisticRegression(max_iter=1000)

for nombre, m in modelos.items():
    pred = m.fit(X_t_tr, y_t_tr).predict(X_t_te)
    print(f"{nombre:<22} exactitud = {accuracy_score(y_t_te, pred):.1%}   "
          f"recall urgentes = {recall_score(y_t_te, pred, zero_division=0):.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. **95.5 %**: exactamente la proporción de tickets **no** urgentes en prueba.
2. **0 %**: nunca dice "urgente", así que no detecta ninguno. `prior` da lo mismo (sus predicciones son iguales; solo cambian sus probabilidades).
3. **`uniform`** (48 %): al lanzar una moneda, "atrapa" a la mitad de los urgentes… a costa de una exactitud de 50 %. `stratified` queda en medio (90.6 % de exactitud, 7 % de recall).

Y lo más importante: la regresión logística logra **96.4 %** de exactitud. Suena genial hasta que ves que la línea base tiene **95.5 %**: el modelo gana **menos de un punto**. Y su recall es solo **25 %**. Sin la línea base, alguien habría presumido "¡96 %!" sin darse cuenta de que el modelo apenas sirve para lo que se quería: encontrar los urgentes.
</details>

### ✋ Ejercicio 5 — Elige la vara
Para cada métrica, ¿qué estrategia de `DummyClassifier` usarías como línea base?
1. Exactitud.
2. Recall de la clase rara.
3. AUC de la curva ROC (Lección 4.3).

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **`most_frequent`**: es la exactitud que obtienes gratis.
2. Para recall, `most_frequent` da 0 (vara demasiado baja) y `uniform` da ~50 %. Lo honesto es reportar **recall junto con precisión** contra una línea base que tenga el mismo nivel de alarmas; por eso en la Lección 4.3 compararemos curvas completas, no un solo número.
3. **`prior`** (o cualquiera constante): todas las probabilidades son iguales, así que su AUC es **0.5**, el valor de "no distingue nada".
</details>

---
## 4. 📖 Líneas base para series de tiempo: "igual que ayer" y "igual que la semana pasada"

Cuando los datos tienen **orden en el tiempo** (Lección 4.6), adivinar el promedio suele ser una línea base demasiado débil. Hay dos clásicas:

| Línea base | Predicción para mañana | Funciona bien cuando… |
|---|---|---|
| **Persistencia** (*naive*) | lo que pasó **hoy** | la serie cambia lento (temperatura, precio de una acción) |
| **Estacional ingenua** (*seasonal naive*) | lo que pasó **el mismo día de la semana pasada** (o del año pasado) | hay ciclos (ventas, tráfico, consumo eléctrico) |

Es el meteorólogo perezoso de la Sección 1, en dos versiones.

Caso: las **ventas diarias de una panadería** durante dos años. Los sábados se vende mucho más que los lunes, y el negocio crece poco a poco.

In [ ]:
rng = np.random.default_rng(4)
dias = pd.date_range("2024-01-01", periods=728, freq="D")
patron_semanal = np.array([90, 95, 100, 105, 140, 180, 120])          # lun … dom
t = np.arange(len(dias))
ventas = patron_semanal[dias.day_of_week] + 0.05 * t + rng.normal(0, 12, len(dias))

pan = pd.DataFrame({"ventas": ventas.round()}, index=dias)
pan["ayer"] = pan.ventas.shift(1)            # lo que se vendió el día anterior
pan["semana_pasada"] = pan.ventas.shift(7)   # lo que se vendió el mismo día, 7 días antes
pan = pan.dropna()

# Las últimas 8 semanas son la "prueba" (el futuro); lo anterior es entrenamiento
corte = pan.index[-56]
tr, te = pan[pan.index < corte], pan[pan.index >= corte]

pan.ventas[-70:].plot(marker=".", title="Ventas diarias de la panadería (últimas 10 semanas)")
plt.axvline(corte, color="gray", ls="--"); plt.show()

### 🔮 Ejercicio 6 — ¿Qué perezoso gana?
Vamos a medir el MAE (Lección 4.1), en las 8 semanas de prueba, de tres líneas base:
- **A)** Predecir siempre el **promedio** del entrenamiento.
- **B)** **Persistencia**: lo mismo que ayer.
- **C)** **Estacional ingenua**: lo mismo que el mismo día de la semana pasada.

Ordénalas de mejor a peor. ¿Crees que "igual que ayer" le gana al promedio?

✍️ *Mi orden:*

In [ ]:
errores = {
    "A) Promedio":            mean_absolute_error(te.ventas, np.full(len(te), tr.ventas.mean())),
    "B) Igual que ayer":      mean_absolute_error(te.ventas, te.ayer),
    "C) Igual que la semana pasada": mean_absolute_error(te.ventas, te.semana_pasada),
}
for nombre, e in sorted(errores.items(), key=lambda kv: kv[1]):
    print(f"{nombre:<32} MAE = {e:5.1f} panes")

<details><summary><b>👀 Ver respuesta</b></summary>

**C (14.4) → A (27.7) → B (30.6)**.

¡"Igual que ayer" es **peor** que el promedio! Con un patrón semanal tan marcado, decir "el sábado venderé lo del viernes" o "el lunes lo del domingo" falla justo en los saltos. La estacional ingenua respeta el ciclo y reduce el error a la mitad.

Moraleja: **elige la línea base que respete la estructura de los datos**. Si hubieras comparado tu modelo solo contra "igual que ayer", cualquier cosa te habría parecido buena.
</details>

### 📖 La regla del experto: otra línea base imprescindible
Además de las líneas base "tontas", compara siempre contra **lo que el negocio ya hace hoy**. La dueña de la panadería, por ejemplo, hornea según esta regla:

> *"Para cada día, horneo el promedio de lo que vendí ese mismo día de la semana en las últimas 4 semanas."*

Si tu modelo no le gana a la regla de la dueña, **no vale la pena** cambiar su forma de trabajar.

### ✋ Ejercicio 7 — ¿Le gana la dueña a la estacional ingenua?
¿Crees que promediar 4 semanas mejorará el MAE de 14.4 de la estacional ingenua? ¿Por qué?

✍️ *Mi apuesta:*

In [ ]:
# Promedio de las ventas del mismo día de la semana, hace 7, 14, 21 y 28 días
pan["regla_duena"] = sum(pan.ventas.shift(7 * k) for k in [1, 2, 3, 4]) / 4
te = pan[pan.index >= corte]
print(f"Regla de la dueña  MAE = {mean_absolute_error(te.ventas, te.regla_duena):5.1f} panes")

<details><summary><b>👀 Ver respuesta</b></summary>

Sí: la regla de la dueña baja el error a **≈10.5 panes**. Al promediar cuatro semanas, el "ruido" de un día raro (una fiesta, un día de lluvia) se diluye. La estacional ingenua copia también ese ruido.

👉 La dueña **ya** tiene una regla bastante buena. Esa es la vara real que tu modelo debe superar.
</details>

---
## 5. 📖 ¿Cuánto mejor? La mejora relativa (*skill score*)

Decir "mi modelo tiene MAE 9.3 y la línea base 14.4" está bien, pero para comparar proyectos distintos conviene un solo número:

$$\text{mejora relativa} = 1 - \frac{\text{error del modelo}}{\text{error de la línea base}}$$

| Valor | Significa |
|---|---|
| **1** | Error cero: perfecto |
| **0.3** | El modelo elimina el 30 % del error de la línea base |
| **0** | Igual que la línea base: el modelo no aporta nada |
| **< 0** | **Peor** que la línea base 🚨 |

> 📝 ¿Te suena? El **R²** (Lección 4.1) es exactamente esta fórmula, usando el error cuadrático y la línea base del promedio.

Entrenemos un modelo de verdad para la panadería: una regresión lineal que usa el día de la semana, la tendencia y las ventas de la semana pasada.

In [ ]:
def caracteristicas(d):
    X = pd.get_dummies(d.index.day_of_week, prefix="dia").astype(float)
    X.index = d.index
    X["tendencia"] = np.arange(len(pan))[pan.index.isin(d.index)]
    X["semana_pasada"] = d.semana_pasada
    return X

tr, te = pan[pan.index < corte].dropna(), pan[pan.index >= corte]
modelo_pan = LinearRegression().fit(caracteristicas(tr), tr.ventas)
mae_modelo = mean_absolute_error(te.ventas, modelo_pan.predict(caracteristicas(te)))
print(f"Modelo  MAE = {mae_modelo:.1f} panes\n")

for nombre, mae_base in {**errores, "D) Regla de la dueña": mean_absolute_error(te.ventas, te.regla_duena)}.items():
    print(f"Mejora relativa contra {nombre:<32}: {1 - mae_modelo / mae_base:+.0%}")

### 🔮 Ejercicio 8 — ¿Qué mejora reportarías?
El modelo tiene un MAE de **9.3**. Mira las cuatro mejoras relativas.
1. Un vendedor de software te presume la mejora del **66 %**. ¿Contra qué línea base la calculó? ¿Es honesta?
2. ¿Cuál reportarías tú al dueño del negocio?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. El 66 % sale de compararse contra el **promedio** (27.7). Y la cifra contra "igual que ayer" es todavía más espectacular (+70 %). Son comparaciones técnicamente ciertas pero **engañosas**: nadie en la panadería hornea "el promedio de dos años".
2. La mejora contra **la regla de la dueña: ≈ +11 %**. Es lo que realmente gana el negocio al cambiar de método. ¿Vale la pena? Depende del costo de implementar el modelo frente al ahorro de equivocarse ~1.2 panes menos al día.

**Regla:** reporta la mejora contra la **línea base más fuerte** que exista, no contra la más débil.
</details>

---
## 6. 📖 Comparar en las mismas condiciones

Una línea base solo sirve si se mide **exactamente igual** que el modelo:
- **Mismo conjunto** de datos (las mismas filas de prueba o los mismos pliegues de validación cruzada).
- **Misma métrica**.
- **Misma información disponible**: si el modelo no puede ver el futuro, la línea base tampoco.

Lo más cómodo es tratar a la línea base como **un modelo más** dentro de la validación cruzada (Lección 4.6):

In [ ]:
for nombre, m in [("Dummy (media)", DummyRegressor()), ("Regresión lineal", LinearRegression())]:
    mae_cv = -cross_val_score(m, X, y, cv=5, scoring="neg_mean_absolute_error")
    print(f"{nombre:<18} MAE por pliegue: {mae_cv.round(1)}   promedio = {mae_cv.mean():.1f}")

### ✋ Ejercicio 9 — Detecta la trampa
Un colega te muestra esta tabla. ¿Qué está mal en cada fila?

| Modelo | MAE | Cómo se midió |
|---|---|---|
| Modelo nuevo | 9.0 | Últimas 8 semanas (incluye diciembre) |
| Línea base | 13.0 | Todo el año 2024 |
| Regla de la dueña | 11.5 | Últimas 8 semanas, pero usando las ventas **del mismo día** (que aún no se conocen) |

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **Fila 1 vs. fila 2:** están medidas en **periodos distintos**. Las semanas de prueba pueden ser más fáciles o más difíciles que el año completo: la comparación no es justa. Hay que medir a todos en las mismas 8 semanas (o en los mismos pliegues).
- **Fila 3:** la regla usa información que **no estaría disponible** al momento de predecir (fuga de datos, Lección 2.6). Su MAE es artificialmente bueno… o, si se lo arreglan, la comparación cambiará.

Una comparación justa: **mismas filas, misma métrica, misma información**.
</details>

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Reportas "95 % de exactitud" y nadie pregunta "¿comparado con qué?" | **No hay línea base** | Agrega siempre `DummyClassifier`/`DummyRegressor` a tu tabla |
| El modelo "mejora un 70 %"… contra el promedio, en una serie con ciclos | **Línea base demasiado débil** | Usa la estacional ingenua o la regla actual del negocio |
| La línea base se midió en otro periodo, otro conjunto u otra métrica | **Comparación injusta** | Mide todo con los mismos pliegues de validación cruzada |
| El modelo le gana a la línea base por 0.5 puntos | Mejora **dentro del ruido** | Mira la variación entre pliegues (Lección 4.6) antes de celebrar |
| Mejora relativa **negativa** | El modelo es peor que no hacer nada | Revisa sobreajuste (Lección 4.4), fuga de datos o la métrica |
| Comparaste solo contra líneas base "tontas" | Te falta la vara real | Pregunta: "¿cómo se decide esto hoy?" y úsalo como línea base |

## 8. 🧠 Resumen en 5 puntos
1. Una **línea base** es la predicción más simple y razonable **sin modelo**; ningún número de evaluación tiene sentido sin ella.
2. En regresión: `DummyRegressor` (media o mediana, según tu métrica). El **R²** ya compara contra la media: R² = 0 es "igual que el promedio".
3. En clasificación: `DummyClassifier`. Con desbalance, `most_frequent` logra exactitudes altísimas sin detectar **nada**.
4. En series de tiempo: **persistencia** y **estacional ingenua**; y en todo proyecto, la **regla actual del negocio**.
5. Reporta la **mejora relativa** contra la línea base **más fuerte**, medida en **las mismas condiciones** que el modelo.

---
## 9. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 9.1 Repaso rápido (opción múltiple)

**1.** Un modelo de regresión tiene R² = −0.15 en prueba. ¿Qué significa?
- a) Explica el 15 % de la variación · b) Es peor que predecir siempre el promedio · c) Es un error de cálculo · d) Es perfecto

**2.** En un hospital, el 2 % de los pacientes tiene una infección grave. ¿Qué exactitud logra, sin modelo, la línea base `most_frequent`?
- a) 2 % · b) 50 % · c) 98 % · d) 100 %

**3.** Vas a reportar el **MAE** de un modelo. ¿Cuál es la línea base constante más justa?
- a) La media · b) La mediana · c) El máximo · d) Cero

**4.** Predices el consumo eléctrico **por hora** de una ciudad. ¿Qué línea base es probablemente la más fuerte?
- a) El promedio histórico · b) La hora anterior · c) La misma hora del mismo día de la semana pasada · d) Una moneda al aire

**5.** Tu modelo tiene MAE 8 y la línea base, MAE 10. ¿Cuál es la mejora relativa?
- a) 2 % · b) 20 % · c) 25 % · d) 80 %

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Un R² negativo significa que el modelo se equivoca **más** que la línea base del promedio. Señal de alerta.
2. **c)** Decir "nadie tiene infección" acierta el 98 %… y no detecta a ningún enfermo.
3. **b)** La mediana es la constante que minimiza el error absoluto.
4. **c)** El consumo tiene ciclos diarios y semanales; la estacional ingenua los respeta. (La hora anterior también es fuerte, porque el consumo cambia lento, pero falla en los cambios bruscos de la mañana y la noche.)
5. **b)** 1 − 8/10 = 0.20 → el modelo elimina el 20 % del error.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Tu jefe dice: *"Para qué perder tiempo con líneas base tontas, si sabemos que nuestro modelo será mejor"*. Da dos razones concretas por las que vale la pena calcularlas **antes** de entrenar el primer modelo.

**B. Caso de análisis 💼** Un hospital usa hoy esta regla: *"Se le hace el examen especial a todo paciente mayor de 60 años con fiebre"*. Un equipo propone un modelo que detecta la misma enfermedad con un recall del 85 %. ¿Qué datos necesitarías para saber si el modelo vale la pena? ¿Contra qué lo compararías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Por ejemplo:
1. **Calibran expectativas**: si `most_frequent` ya da 98 %, sabrás desde el inicio que la exactitud no sirve como métrica y elegirás otra (Lecciones 4.2 y 4.8).
2. **Detectan errores**: si tu primer modelo sale *peor* que la línea base, hay un bug, fuga de datos o un problema serio, y lo descubres temprano.
3. (Extra) Son un **plan B** gratis: a veces la línea base es "suficientemente buena" y el proyecto de ML no hace falta.

**B.** La línea base correcta es **la regla actual del hospital**, medida sobre los **mismos pacientes**: ¿qué recall y qué precisión tiene la regla "mayor de 60 + fiebre"? ¿A cuántos pacientes manda al examen (costo)? El modelo vale la pena si, con el mismo número de exámenes, detecta más enfermos, o si detecta los mismos con menos exámenes. Un recall de 85 % aislado no dice nada: la regla podría tener 90 %.
</details>

### 9.3 🧩 Práctica integradora: "¿Le compramos el modelo al proveedor?"

Una app de comida a domicilio quiere predecir **cuántos minutos tardará cada pedido**. Un proveedor les ofrece un modelo y presume:

> 📢 *"Nuestro modelo predice el tiempo de entrega con un error promedio (MAE) de solo **6 minutos**."*

Hoy, el despachador usa una regla: **"20 minutos + 3 minutos por cada km, y 10 minutos extra en hora pico"**.

¿Vale la pena pagar por el modelo? Lo resolverás en 6 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Mira los datos.** Tenemos 1,500 pedidos históricos.

In [ ]:
rng_ped = np.random.default_rng(11)
n = 1_500
pedidos = pd.DataFrame({
    "distancia_km": rng_ped.uniform(0.5, 12, n).round(1),
    "articulos":    rng_ped.integers(1, 15, n),
    "hora_pico":    rng_ped.integers(0, 2, n),             # 1 = sí
    "lluvia":       (rng_ped.random(n) < 0.2).astype(int), # 1 = sí
})
pedidos["minutos"] = (12 + 2.6 * pedidos.distancia_km + 0.6 * pedidos.articulos + 8 * pedidos.hora_pico
                      + 7 * pedidos.lluvia + rng_ped.gamma(2, 3, n)).round(1)

X_p, y_p = pedidos.drop(columns="minutos"), pedidos.minutos
X_p_tr, X_p_te, y_p_tr, y_p_te = train_test_split(X_p, y_p, test_size=0.25, random_state=0)
print(f"Tiempo promedio de entrega: {y_p.mean():.1f} min")
pedidos.head()

**Paso 2 🔮 — La línea base tonta.** Un pedido tarda en promedio ~44 minutos, con bastante variación. Si predijeras siempre la **mediana**, ¿qué MAE esperas: cerca de 2, de 9 o de 20 minutos?

✍️ *Mi apuesta:*

In [ ]:
base = DummyRegressor(strategy="median").fit(X_p_tr, y_p_tr)
mae_tonta = mean_absolute_error(y_p_te, base.predict(X_p_te))
print(f"Línea base (mediana = {base.predict(X_p_te[:1])[0]} min)  →  MAE = {mae_tonta:.1f} min")

<details><summary><b>👀 Ver respuesta</b></summary>

**≈ 9.0 minutos.** Sin mirar nada del pedido, solo diciendo "44.6 minutos" a todos, te equivocas ~9 minutos en promedio. El 6 del proveedor le gana a esto… pero esa no es la vara importante.
</details>

**Paso 3 ✋ — La línea base del negocio.** Completa la regla del despachador en la celda (ya está escrita; revísala) y calcula su MAE **sobre los mismos pedidos de prueba**.

In [ ]:
regla = 20 + 3 * X_p_te.distancia_km + 10 * X_p_te.hora_pico     # ✏️ la regla del despachador
mae_regla = mean_absolute_error(y_p_te, regla)
print(f"Regla del despachador  →  MAE = {mae_regla:.1f} min")

<details><summary><b>👀 Ver respuesta</b></summary>

**≈ 4.6 minutos.** ¡La regla que ya usa el despachador es **mejor** que el modelo del proveedor (6 min)! Pagar por ese modelo empeoraría las estimaciones.

Con la mejora relativa: 1 − 6 / 4.6 ≈ **−30 %**. Negativa: 🚨.
</details>

**Paso 4 ▶️ — ¿Podemos hacerlo mejor nosotros?** Entrenamos una regresión lineal sencilla con las 4 columnas.

In [ ]:
mi_modelo = LinearRegression().fit(X_p_tr, y_p_tr)
mae_mio = mean_absolute_error(y_p_te, mi_modelo.predict(X_p_te))
print(f"Nuestra regresión lineal  →  MAE = {mae_mio:.1f} min")

**Paso 5 ✋ — Calcula las mejoras relativas.** Completa la tabla a mano (o en la celda) con la fórmula 1 − error_modelo / error_base:

| Modelo | contra mediana (9.0) | contra regla (4.6) |
|---|---|---|
| Proveedor (6.0) | ? | ? |
| Nuestra regresión | ? | ? |

In [ ]:
for nombre, e in [("Proveedor", 6.0), ("Nuestra regresión", mae_mio)]:
    print(f"{nombre:<18} vs mediana: {1 - e / mae_tonta:+.0%}    vs regla: {1 - e / mae_regla:+.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **Proveedor:** +33 % contra la mediana (lo que seguramente presumirá en su presentación), pero **−30 %** contra la regla del despachador.
- **Nuestra regresión (MAE ≈ 3.5):** +61 % contra la mediana y **+25 %** contra la regla: le quita una cuarta parte del error a lo que ya se hace hoy.
</details>

**Paso 6 ✋ — Tu recomendación.** Escribe en 3 frases qué le dirías al gerente. Menciona: qué línea base usaste, si compraría el modelo del proveedor y qué condición le pondrías a cualquier modelo futuro.

✍️ *Mi recomendación:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un ejemplo:

*"No recomiendo comprar el modelo del proveedor: su error promedio (6 min) es peor que el de la regla que ya usa nuestro despachador (4.6 min). Un modelo sencillo entrenado con nuestros propios datos baja el error a 3.5 min, una mejora del 25 % sobre la regla actual. A cualquier proveedor le pediremos que mida su modelo con **nuestros** pedidos y lo compare contra la regla del despachador, en las mismas condiciones."*

Fíjate en que el número del proveedor (6 min) **no era falso**: simplemente nadie lo había puesto junto a la vara correcta.
</details>

---
## 📚 Para profundizar
- 📘 **Documentación:** scikit-learn, guía de usuario, sección *Dummy estimators*.
- 📗 **Libro:** Hyndman y Athanasopoulos, *Forecasting: Principles and Practice* (gratis en línea), capítulo "Some simple forecasting methods" (persistencia, estacional ingenua, *skill scores*).
- 📺 **Video:** busca "baseline models machine learning" — cualquier charla sobre por qué empezar por una línea base.

---
⬅️ **Anterior:** [4.6 Validación cruzada: k-fold, estratificada y temporal](4.6-validacion-cruzada.ipynb) · ➡️ **Siguiente:** [4.8 Elegir la métrica según el costo del error en el negocio](4.8-metrica-segun-costo.ipynb)